# Windblade YOLO11n train/validation apparatus
This output-free notebook runs the three frozen class-agnostic seeds. It uses only training and validation data; the held-out split remains sealed. Run cells in order on a Colab GPU runtime with at least 8 GiB VRAM.

In [1]:
import sys

EXPECTED_PYTHON = (3, 11)

if sys.version_info[:2] != EXPECTED_PYTHON:
    raise RuntimeError(
        "Phase 11B requires Colab runtime version 2025.07 "
        "(Python 3.11). Select Runtime > Change runtime type > "
        "Runtime version: 2025.07, choose T4 GPU, reconnect, "
        "and restart from this cell."
    )

print("Python version gate passed:", sys.version)

Python version gate passed: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]


In [2]:
from google.colab import drive
drive.mount('/content/drive')
REPOSITORY_URL = 'https://github.com/Stukhori/Bade-defect-recognition.git'
APPARATUS_COMMIT = 'e2bc2f76b3882b80bc8dd7e8cca54a12e2e4d272'
REPOSITORY_ROOT = '/content/Bade-defect-recognition'
DRIVE_ROOT = '/content/drive/MyDrive/windblade_phase11b'
DATA_ROOT = '/content/windblade_phase11b_data'

Mounted at /content/drive


In [3]:
import pathlib, shutil, subprocess
if APPARATUS_COMMIT == 'REPLACE_WITH_APPARATUS_COMMIT':
    raise ValueError('Set the full apparatus commit before continuing')
if pathlib.Path(REPOSITORY_ROOT).exists():
    shutil.rmtree(REPOSITORY_ROOT)
subprocess.run(['git', 'clone', '--filter=blob:none', REPOSITORY_URL, REPOSITORY_ROOT], check=True)
subprocess.run(['git', 'checkout', '--detach', APPARATUS_COMMIT], cwd=REPOSITORY_ROOT, check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPOSITORY_ROOT, text=True).strip()
assert head == APPARATUS_COMMIT

In [4]:
subprocess.run(['python', '-m', 'pip', 'install', '--requirement', f'{REPOSITORY_ROOT}/requirements-detection-colab.txt'], check=True)

CompletedProcess(args=['python', '-m', 'pip', 'install', '--requirement', '/content/Bade-defect-recognition/requirements-detection-colab.txt'], returncode=0)

In [5]:
subprocess.run(['python', '-m', 'pip', 'install', '--no-deps', '--editable', REPOSITORY_ROOT], check=True)

CompletedProcess(args=['python', '-m', 'pip', 'install', '--no-deps', '--editable', '/content/Bade-defect-recognition'], returncode=0)

In [6]:
BASE = ['python', 'scripts/run_phase11b.py', '--drive-root', DRIVE_ROOT, '--data-root', DATA_ROOT]
subprocess.run(BASE + ['apparatus-check'], cwd=REPOSITORY_ROOT, check=True)
subprocess.run(BASE + ['verify-archive'], cwd=REPOSITORY_ROOT, check=True)
subprocess.run(BASE + ['preflight'], cwd=REPOSITORY_ROOT, check=True)

CompletedProcess(args=['python', 'scripts/run_phase11b.py', '--drive-root', '/content/drive/MyDrive/windblade_phase11b', '--data-root', '/content/windblade_phase11b_data', 'preflight'], returncode=0)

In [7]:
record = pathlib.Path(DRIVE_ROOT) / 'provenance/phase11b_weight_acquisition.json'
if not record.exists():
    subprocess.run(BASE + ['acquire-weight', '--apparatus-commit', APPARATUS_COMMIT], cwd=REPOSITORY_ROOT, check=True)
else:
    print('Using the existing immutable weight-acquisition record; training will verify its bytes.')

Using the existing immutable weight-acquisition record; training will verify its bytes.


In [8]:
subprocess.run(BASE + ['materialize-trainval'], cwd=REPOSITORY_ROOT, check=True)

CompletedProcess(args=['python', 'scripts/run_phase11b.py', '--drive-root', '/content/drive/MyDrive/windblade_phase11b', '--data-root', '/content/windblade_phase11b_data', 'materialize-trainval'], returncode=0)

In [10]:
from pathlib import Path

drive = Path(DRIVE_ROOT)
run = drive / "runs" / "seed_17"

print("SEED 17 DIRECTORY EXISTS:", run.exists())
print("SEED 29 DIRECTORY EXISTS:", (drive / "runs" / "seed_29").exists())
print("SEED 43 DIRECTORY EXISTS:", (drive / "runs" / "seed_43").exists())

print(
    "WEIGHT RECORD EXISTS:",
    (drive / "provenance" / "phase11b_weight_acquisition.json").is_file(),
)
print(
    "INITIAL WEIGHT EXISTS:",
    (drive / "provenance" / "yolo11n.pt").is_file(),
)
print(
    "MATERIALIZATION RECORD EXISTS:",
    (Path(DATA_ROOT) / "materialization.json").is_file(),
)

if run.exists():
    print("\nSEED 17 FILES:")
    for path in sorted(run.rglob("*")):
        if path.is_file():
            print(path.relative_to(run), "SIZE:", path.stat().st_size)

    for relative in ("run_state.json", "args.yaml", "results.csv"):
        path = run / relative
        if path.is_file():
            print(f"\n===== {relative} =====")
            print(path.read_text(encoding="utf-8", errors="replace"))

SEED 17 DIRECTORY EXISTS: True
SEED 29 DIRECTORY EXISTS: True
SEED 43 DIRECTORY EXISTS: False
WEIGHT RECORD EXISTS: True
INITIAL WEIGHT EXISTS: True
MATERIALIZATION RECORD EXISTS: True

SEED 17 FILES:
F1_curve.png SIZE: 112823
PR_curve.png SIZE: 87298
P_curve.png SIZE: 102073
R_curve.png SIZE: 108866
args.yaml SIZE: 1747
confusion_matrix.png SIZE: 89280
confusion_matrix_normalized.png SIZE: 93875
labels.jpg SIZE: 222451
labels_correlogram.jpg SIZE: 241965
results.csv SIZE: 12707
results.png SIZE: 300698
run_state.json SIZE: 508
train_batch0.jpg SIZE: 356189
train_batch1.jpg SIZE: 356807
train_batch2.jpg SIZE: 361789
train_batch2880.jpg SIZE: 308324
train_batch2881.jpg SIZE: 298376
train_batch2882.jpg SIZE: 300942
val_batch0_labels.jpg SIZE: 407013
val_batch0_pred.jpg SIZE: 417252
val_batch1_labels.jpg SIZE: 377699
val_batch1_pred.jpg SIZE: 416725
val_batch2_labels.jpg SIZE: 335599
val_batch2_pred.jpg SIZE: 342206
weights/best.pt SIZE: 5477331
weights/epoch0.pt SIZE: 16075540
weights/ep

In [12]:
from pathlib import Path
import csv
import hashlib
import json
import yaml

runs_root = Path(DRIVE_ROOT) / "runs"

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

for seed in (17, 29, 43):
    run_dir = runs_root / f"seed_{seed}"
    state_path = run_dir / "run_state.json"
    results_path = run_dir / "results.csv"
    args_path = run_dir / "args.yaml"
    last_path = run_dir / "weights" / "last.pt"

    print(f"\n========== SEED {seed} ==========")
    print("RUN DIRECTORY:", run_dir.exists())

    if state_path.is_file():
        state = json.loads(state_path.read_text())
        print("STATE:", {
            key: state.get(key)
            for key in (
                "seed",
                "status",
                "configuration_sha256",
                "apparatus_commit",
                "applied_batch_size",
                "applied_optimizer",
                "applied_amp",
                "last_checkpoint_sha256",
            )
        })
    else:
        state = {}
        print("STATE: MISSING")

    if results_path.is_file():
        with results_path.open(newline="") as handle:
            rows = list(csv.DictReader(handle))
        print("RESULT ROWS:", len(rows))
        print("LAST RECORDED EPOCH:", rows[-1].get("epoch") if rows else None)
    else:
        print("RESULTS: MISSING")

    if args_path.is_file():
        args = yaml.safe_load(args_path.read_text())
        print("TRAINING ARGS:", {
            key: args.get(key)
            for key in (
                "data",
                "epochs",
                "batch",
                "optimizer",
                "lr0",
                "cos_lr",
                "seed",
                "resume",
            )
        })
    else:
        print("ARGS: MISSING")

    epoch_files = list((run_dir / "weights").glob("epoch*.pt"))
    print("EPOCH CHECKPOINT COUNT:", len(epoch_files))
    print("LAST.PT EXISTS:", last_path.is_file())

    if last_path.is_file():
        observed_sha = sha256(last_path)
        print("LAST.PT SHA256:", observed_sha)
        print(
            "STATE HASH MATCH:",
            state.get("last_checkpoint_sha256") == observed_sha
            if state.get("last_checkpoint_sha256")
            else "NO HASH RECORDED"
        )


========== SEED 17 ==========
RUN DIRECTORY: True
STATE: {'seed': 17, 'status': None, 'configuration_sha256': 'fc0ab33a25bafb5b92da88f67343bca9bbcecb6c715d867b06f4ac74f90cff1b', 'apparatus_commit': 'e2bc2f76b3882b80bc8dd7e8cca54a12e2e4d272', 'applied_batch_size': None, 'applied_optimizer': None, 'applied_amp': None, 'last_checkpoint_sha256': None}
RESULT ROWS: 100
LAST RECORDED EPOCH: 100
TRAINING ARGS: {'data': '/content/windblade_phase11b_data/dataset/trainval.yaml', 'epochs': 100, 'batch': 16, 'optimizer': 'AdamW', 'lr0': 0.001, 'cos_lr': True, 'seed': 17, 'resume': '/content/drive/MyDrive/windblade_phase11b/runs/seed_17/weights/last.pt'}
EPOCH CHECKPOINT COUNT: 100
LAST.PT EXISTS: True
LAST.PT SHA256: b588d29a3d03ad4315ed7d6a3fb8c8c0875eaf193b2614e1b2972987b1b2cb99
STATE HASH MATCH: NO HASH RECORDED

========== SEED 29 ==========
RUN DIRECTORY: True
STATE: {'seed': 29, 'status': None, 'configuration_sha256': 'fc0ab33a25bafb5b92da88f67343bca9bbcecb6c715d867b06f4ac74f90cff1b', 'appa

In [13]:
from pathlib import Path
import hashlib
import subprocess

RECOVERY_COMMIT = "749cc29f81b5223edd36dddc3928b1814d029096"
EXPECTED_CONFIG_SHA256 = (
    "fc0ab33a25bafb5b92da88f67343bca9bbcecb6c715d867b06f4ac74f90cff1b"
)

root = Path(REPOSITORY_ROOT)
config_path = root / "configs" / "detection_phase11b.yaml"

def git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=root,
        text=True,
        capture_output=True,
        check=True,
    ).stdout.strip()

initial_status = git("status", "--porcelain")
print("OLD HEAD:", git("rev-parse", "HEAD"))
print("INITIAL STATUS:", repr(initial_status))

assert initial_status == "", "STOP: repository is not clean"

git("fetch", "origin", "main")
git("cat-file", "-e", RECOVERY_COMMIT + "^{commit}")
git("checkout", "--detach", RECOVERY_COMMIT)

config_sha256 = hashlib.sha256(config_path.read_bytes()).hexdigest()
final_status = git("status", "--porcelain")

print("NEW HEAD:", git("rev-parse", "HEAD"))
print("CONFIG SHA256:", config_sha256)
print("FINAL STATUS:", repr(final_status))

assert git("rev-parse", "HEAD") == RECOVERY_COMMIT
assert config_sha256 == EXPECTED_CONFIG_SHA256
assert final_status == ""

print("RECOVERY COMMIT CHECKOUT: PASS")

OLD HEAD: e2bc2f76b3882b80bc8dd7e8cca54a12e2e4d272
INITIAL STATUS: ''
NEW HEAD: 749cc29f81b5223edd36dddc3928b1814d029096
CONFIG SHA256: fc0ab33a25bafb5b92da88f67343bca9bbcecb6c715d867b06f4ac74f90cff1b
FINAL STATUS: ''
RECOVERY COMMIT CHECKOUT: PASS


In [14]:
import subprocess

BASE = [
    "python",
    "scripts/run_phase11b.py",
    "--drive-root", DRIVE_ROOT,
    "--data-root", DATA_ROOT,
]

for command in ("apparatus-check", "verify-archive", "preflight"):
    result = subprocess.run(
        BASE + [command],
        cwd=REPOSITORY_ROOT,
        text=True,
        capture_output=True,
    )

    print(f"\n========== {command} ==========")
    print("RETURN CODE:", result.returncode)
    print("STDOUT:")
    print(result.stdout)
    print("STDERR:")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"{command} failed; stop here")

print("\nALL PRE-TRAINING CHECKS: PASS")


========== apparatus-check ==========
RETURN CODE: 0
STDOUT:
{
  "boxes": 1065,
  "boxes_by_split": {
    "test": 162,
    "train": 757,
    "validation": 146
  },
  "config_fingerprint": "9f4a20ba4404c9a6072277a504c466a0756143b908e79c7168d2ccf91ff32057",
  "dataset_fingerprint": "ad4ab59c3e3c85c6cf0b85b148177bd6b79d24f372f49bdff0043609e6fefc97",
  "images": 720,
  "images_by_split": {
    "test": 109,
    "train": 510,
    "validation": 101
  },
  "run_count": 3,
  "scientific_output_fingerprint": "3f46cbdc6c7a2e3cf6093ff177dd1948d113fa4c36fa9eb907d7c8621e800461",
  "seeds": [
    17,
    29,
    43
  ],
  "split_fingerprint": "264f8460f203074374c2c098c8fd5d2e55fb7ee1f281a8d505e2dfb0de9a2bc3",
  "status": "PASS"
}

STDERR:


========== verify-archive ==========
RETURN CODE: 0
STDOUT:
{
  "filename": "WT blade defect dataset.zip",
  "sha256": "466452f2a0cfc9ef6ba63ea2a3bbc7ea4262057dd07e4fc9e00eedf5bba305b4",
  "size_bytes": 78958553,
  "status": "PASS"
}

STDERR:


========== preflig

In [15]:
from pathlib import Path
import hashlib
import json
import subprocess

RECOVERY_COMMIT = "749cc29f81b5223edd36dddc3928b1814d029096"
run_dir = Path(DRIVE_ROOT) / "runs" / "seed_17"
weights_dir = run_dir / "weights"

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

protected_files = [
    run_dir / "results.csv",
    run_dir / "args.yaml",
    weights_dir / "best.pt",
    weights_dir / "last.pt",
]

before_hashes = {str(path): sha256(path) for path in protected_files}
before_epoch_metadata = {
    path.name: (path.stat().st_size, path.stat().st_mtime_ns)
    for path in sorted(weights_dir.glob("epoch*.pt"))
}

BASE = [
    "python",
    "scripts/run_phase11b.py",
    "--drive-root", DRIVE_ROOT,
    "--data-root", DATA_ROOT,
]

result = subprocess.run(
    BASE + ["train", "--seed", "17"],
    cwd=REPOSITORY_ROOT,
    text=True,
    capture_output=True,
)

print("RETURN CODE:", result.returncode)
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Seed 17 recovery failed; stop here")

after_hashes = {str(path): sha256(path) for path in protected_files}
after_epoch_metadata = {
    path.name: (path.stat().st_size, path.stat().st_mtime_ns)
    for path in sorted(weights_dir.glob("epoch*.pt"))
}

state = json.loads((run_dir / "run_state.json").read_text())

print("STATUS:", state.get("status"))
print("LAST CHECKPOINT SHA256:", state.get("last_checkpoint_sha256"))
print("APPLIED BATCH:", state.get("applied_batch_size"))
print("APPLIED OPTIMIZER:", state.get("applied_optimizer"))
print("APPLIED AMP:", state.get("applied_amp"))
print("RECOVERY:", state.get("completion_recovery"))
print("PROTECTED HASHES UNCHANGED:", before_hashes == after_hashes)
print(
    "EPOCH CHECKPOINTS UNCHANGED:",
    before_epoch_metadata == after_epoch_metadata,
)

assert state["status"] == "TRAINING_COMMAND_COMPLETED"
assert state["last_checkpoint_sha256"] == sha256(weights_dir / "last.pt")
assert state["applied_batch_size"] == 16
assert state["applied_optimizer"] == "AdamW"
assert state["applied_amp"] is True
assert state["completion_recovery"]["training_invoked"] is False
assert state["completion_recovery"]["repository_commit"] == RECOVERY_COMMIT
assert before_hashes == after_hashes
assert before_epoch_metadata == after_epoch_metadata

print("SEED 17 SAFE RECOVERY: PASS")

RETURN CODE: 1
STDOUT:

STDERR:
Traceback (most recent call last):
  File "/content/Bade-defect-recognition/scripts/run_phase11b.py", line 85, in <module>
    raise SystemExit(main())
                     ^^^^^^
  File "/content/Bade-defect-recognition/scripts/run_phase11b.py", line 69, in main
    result = train_seed(config, config_path, repo, data_root, drive_root, weight, weight_record, args.seed)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Bade-defect-recognition/src/windblade/detection/phase11b.py", line 719, in train_seed
    recovered = recover_legacy_completed_run(config, repo, data_root, layout, run_dir, seed, state, expected_state)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Bade-defect-recognition/src/windblade/detection/phase11b.py", line 658, in recover_legacy_completed_run
    applied = _critical_training

RuntimeError: Seed 17 recovery failed; stop here

In [17]:
from pathlib import Path
import hashlib
import subprocess

NEW_COMMIT = "764dd67b31616fa0165a1fe4058b82eb978e4178"
EXPECTED_CONFIG_SHA256 = (
    "fc0ab33a25bafb5b92da88f67343bca9bbcecb6c715d867b06f4ac74f90cff1b"
)

root = Path(REPOSITORY_ROOT)
config_path = root / "configs" / "detection_phase11b.yaml"

def git(*args):
    return subprocess.run(
        ["git", *args],
        cwd=root,
        text=True,
        capture_output=True,
        check=True,
    ).stdout.strip()

print("OLD HEAD:", git("rev-parse", "HEAD"))
print("INITIAL STATUS:", repr(git("status", "--porcelain")))

assert git("status", "--porcelain") == "", "STOP: repository is not clean"

git("fetch", "origin", "main")
git("cat-file", "-e", NEW_COMMIT + "^{commit}")
git("checkout", "--detach", NEW_COMMIT)

config_sha = hashlib.sha256(config_path.read_bytes()).hexdigest()

print("NEW HEAD:", git("rev-parse", "HEAD"))
print("CONFIG SHA256:", config_sha)
print("FINAL STATUS:", repr(git("status", "--porcelain")))

assert git("rev-parse", "HEAD") == NEW_COMMIT
assert config_sha == EXPECTED_CONFIG_SHA256
assert git("status", "--porcelain") == ""

print("DEVICE-NORMALIZATION COMMIT CHECKOUT: PASS")

OLD HEAD: 749cc29f81b5223edd36dddc3928b1814d029096
INITIAL STATUS: ''
NEW HEAD: 764dd67b31616fa0165a1fe4058b82eb978e4178
CONFIG SHA256: fc0ab33a25bafb5b92da88f67343bca9bbcecb6c715d867b06f4ac74f90cff1b
FINAL STATUS: ''
DEVICE-NORMALIZATION COMMIT CHECKOUT: PASS


In [18]:
from pathlib import Path
import hashlib
import json
import subprocess

RECOVERY_COMMIT = "764dd67b31616fa0165a1fe4058b82eb978e4178"
run_dir = Path(DRIVE_ROOT) / "runs" / "seed_17"
weights_dir = run_dir / "weights"

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

protected_files = [
    run_dir / "results.csv",
    run_dir / "args.yaml",
    weights_dir / "best.pt",
    weights_dir / "last.pt",
]

before_hashes = {str(path): sha256(path) for path in protected_files}
before_epoch_metadata = {
    path.name: (path.stat().st_size, path.stat().st_mtime_ns)
    for path in sorted(weights_dir.glob("epoch*.pt"))
}

BASE = [
    "python",
    "scripts/run_phase11b.py",
    "--drive-root", DRIVE_ROOT,
    "--data-root", DATA_ROOT,
]

result = subprocess.run(
    BASE + ["train", "--seed", "17"],
    cwd=REPOSITORY_ROOT,
    text=True,
    capture_output=True,
)

print("RETURN CODE:", result.returncode)
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Seed 17 recovery failed; stop here")

after_hashes = {str(path): sha256(path) for path in protected_files}
after_epoch_metadata = {
    path.name: (path.stat().st_size, path.stat().st_mtime_ns)
    for path in sorted(weights_dir.glob("epoch*.pt"))
}

state = json.loads((run_dir / "run_state.json").read_text())

print("STATUS:", state.get("status"))
print("LAST CHECKPOINT SHA256:", state.get("last_checkpoint_sha256"))
print("APPLIED BATCH:", state.get("applied_batch_size"))
print("APPLIED OPTIMIZER:", state.get("applied_optimizer"))
print("APPLIED AMP:", state.get("applied_amp"))
print("RECOVERY:", state.get("completion_recovery"))
print("PROTECTED HASHES UNCHANGED:", before_hashes == after_hashes)
print("EPOCH CHECKPOINTS UNCHANGED:", before_epoch_metadata == after_epoch_metadata)

assert state["status"] == "TRAINING_COMMAND_COMPLETED"
assert state["last_checkpoint_sha256"] == sha256(weights_dir / "last.pt")
assert state["applied_batch_size"] == 16
assert state["applied_optimizer"] == "AdamW"
assert state["applied_amp"] is True
assert state["completion_recovery"]["training_invoked"] is False
assert state["completion_recovery"]["repository_commit"] == RECOVERY_COMMIT
assert before_hashes == after_hashes
assert before_epoch_metadata == after_epoch_metadata

print("SEED 17 SAFE RECOVERY: PASS")

RETURN CODE: 0
STDOUT:
{
  "apparatus_commit": "e2bc2f76b3882b80bc8dd7e8cca54a12e2e4d272",
  "applied_amp": true,
  "applied_batch_size": 16,
  "applied_optimizer": "AdamW",
  "completion_recovery": {
    "mode": "VERIFIED_PRE_EXISTING_ARTIFACTS",
    "repository_commit": "764dd67b31616fa0165a1fe4058b82eb978e4178",
    "training_invoked": false
  },
  "configuration_path": "configs/detection_phase11b.yaml",
  "configuration_sha256": "fc0ab33a25bafb5b92da88f67343bca9bbcecb6c715d867b06f4ac74f90cff1b",
  "last_checkpoint_sha256": "b588d29a3d03ad4315ed7d6a3fb8c8c0875eaf193b2614e1b2972987b1b2cb99",
  "materialization_fingerprint": "dd5ceb31ef32e9f9af2adeeaa9d3ebcf263eb232621d924845ff2a53785fc011",
  "seed": 17,
  "status": "TRAINING_COMMAND_COMPLETED",
  "weight_path": "/content/drive/MyDrive/windblade_phase11b/provenance/yolo11n.pt",
  "weight_sha256": "0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1"
}

STDERR:

STATUS: TRAINING_COMMAND_COMPLETED
LAST CHECKPOINT SHA256: b

In [19]:
import os
import subprocess

BASE = [
    "python",
    "scripts/run_phase11b.py",
    "--drive-root", DRIVE_ROOT,
    "--data-root", DATA_ROOT,
]

environment = dict(
    os.environ,
    PYTHONHASHSEED="29",
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
)

subprocess.run(
    BASE + ["train", "--seed", "29"],
    cwd=REPOSITORY_ROOT,
    env=environment,
    check=True,
)

CompletedProcess(args=['python', 'scripts/run_phase11b.py', '--drive-root', '/content/drive/MyDrive/windblade_phase11b', '--data-root', '/content/windblade_phase11b_data', 'train', '--seed', '29'], returncode=0)

In [20]:
from pathlib import Path
import csv
import hashlib
import json

run_dir = Path(DRIVE_ROOT) / "runs" / "seed_29"
state_path = run_dir / "run_state.json"
results_path = run_dir / "results.csv"
last_path = run_dir / "weights" / "last.pt"

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

state = json.loads(state_path.read_text())

with results_path.open(newline="") as handle:
    rows = list(csv.DictReader(handle))

observed_last_sha = sha256(last_path)

print("STATUS:", state.get("status"))
print("RESULT ROWS:", len(rows))
print("LAST RECORDED EPOCH:", rows[-1].get("epoch"))
print("APPLIED BATCH:", state.get("applied_batch_size"))
print("APPLIED OPTIMIZER:", state.get("applied_optimizer"))
print("APPLIED AMP:", state.get("applied_amp"))
print("RECORDED LAST SHA:", state.get("last_checkpoint_sha256"))
print("OBSERVED LAST SHA:", observed_last_sha)
print(
    "CHECKPOINT COUNT:",
    len(list((run_dir / "weights").glob("epoch*.pt"))),
)

assert state["status"] == "TRAINING_COMMAND_COMPLETED"
assert state["seed"] == 29
assert state["applied_batch_size"] == 16
assert state["applied_optimizer"] == "AdamW"
assert state["applied_amp"] is True
assert state["last_checkpoint_sha256"] == observed_last_sha
assert len(rows) > 30

print("SEED 29 COMPLETION: PASS")

STATUS: TRAINING_COMMAND_COMPLETED
RESULT ROWS: 100
LAST RECORDED EPOCH: 100
APPLIED BATCH: 16
APPLIED OPTIMIZER: AdamW
APPLIED AMP: True
RECORDED LAST SHA: f5f0640af079ef5a94404c1d8e17cf81b44f7b287130fc2b85c053199d69f667
OBSERVED LAST SHA: f5f0640af079ef5a94404c1d8e17cf81b44f7b287130fc2b85c053199d69f667
CHECKPOINT COUNT: 100
SEED 29 COMPLETION: PASS


In [ ]:
import os
import subprocess

BASE = [
    "python",
    "scripts/run_phase11b.py",
    "--drive-root", DRIVE_ROOT,
    "--data-root", DATA_ROOT,
]

environment = dict(
    os.environ,
    PYTHONHASHSEED="43",
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
)

subprocess.run(
    BASE + ["train", "--seed", "43"],
    cwd=REPOSITORY_ROOT,
    env=environment,
    check=True,
)

In [1]:
subprocess.run(BASE + ['select-validation'], cwd=REPOSITORY_ROOT, check=True)
subprocess.run(BASE + ['bundle'], cwd=REPOSITORY_ROOT, check=True)

NameError: name 'subprocess' is not defined

Stop here. Inspect the generated selection receipt, commit and push it from an authenticated checkout, and follow `docs/phase11b_colab.md` for the separately gated final evaluation. Do not change the frozen configuration or selections.